## Course ID: AIMLZG565
## Course Name: Machine Learning

## Assignment 2
### Name: Raghu Prashanka
### Student ID: 2026AJ05947

Five classifiers (Logistic Regression, Decision Tree, KNN, Gaussian Naive Bayes, SVM) trained on the same split and compared on Accuracy, AUC, Precision, Recall, F1 and MCC.
The training logic lives in `train_models.py`; this notebook runs it and shows the results.

In [ ]:
import platform, sklearn, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns, joblib, json
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

## 1. Dataset

In [ ]:
df = pd.read_csv("data/spambase.csv")
print("Shape:", df.shape)
print("Missing values:", int(df.isna().sum().sum()), "| exact duplicates:", int(df.duplicated().sum()))
print(df["is_spam"].value_counts().rename({0: "not spam", 1: "spam"}))
df.describe().T.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df["is_spam"].map({0: "not spam", 1: "spam"}).value_counts().plot.bar(ax=ax[0], color=["#1f6f8b", "#d1495b"])
ax[0].set_title("Class balance"); ax[0].tick_params(axis="x", rotation=0)
top = df.corr(numeric_only=True)["is_spam"].drop("is_spam").sort_values(key=abs, ascending=False).head(10)
top.plot.barh(ax=ax[1], color="#1f6f8b"); ax[1].invert_yaxis(); ax[1].set_title("Top-10 features by |correlation| with spam")
plt.tight_layout(); plt.show()

## 2. Train, tune (5-fold CV) and evaluate all five models

In [ ]:
import train_models
train_models.main()

## 3. Comparison table (held-out test split)

In [ ]:
metrics = pd.read_csv("model/metrics.csv").set_index("ML Model Name")
metrics.style.format("{:.4f}").highlight_max(axis=0, color="#cfe8d5")

In [ ]:
print(json.dumps(json.load(open("model/best_params.json")), indent=1))

## 4. Confusion matrices of the saved models on `test_data.csv`

In [ ]:
from sklearn.metrics import confusion_matrix
meta = json.load(open("model/meta.json"))
test = pd.read_csv("test_data.csv")
X_test, y_test = test[meta["features"]], test[meta["target"]]
fig, axes = plt.subplots(1, 5, figsize=(20, 3.6))
for ax, (name, f) in zip(axes, meta["models"].items()):
    m = joblib.load(f"model/{f}")
    sns.heatmap(confusion_matrix(y_test, m.predict(X_test)), annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["ham", "spam"], yticklabels=["ham", "spam"])
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()